# Explaining Topological Models with Shapley Games

In this tutorial we use `topobench.explain` to answer two practical questions about a TopoTune ([GCCN](https://arxiv.org/pdf/2410.06530)) model:

1. **Which cells drive this prediction?** The cells of the complex (nodes, edges, faces) are the *players* of a cooperative game: masking the features of absent cells and reading the model's output defines the game, and exact Shapley values attribute the prediction to the cells.
2. **Which neighborhoods earn their keep?** The message-passing neighborhoods of the backbone are the players: masking their routes inside the backbone defines a *performance* game, sampled Shapley values attribute the metric to neighborhoods, and the backbone can then be pruned to a selected coalition and keep training with its trained weights.

In both cases a coalition is encoded as an integer bitmask (bit $i$ = player $i$), a game is any function `v(mask) -> float`, and every attribution can be audited with the executable axiom checks in `topobench.explain.axioms`.

Everything below runs on a toy complex in a few seconds on CPU — no downloads required.

## Table of contents

&emsp;[- Section 1:](#sec1) A toy complex, a small TopoTune model, and a toy task

&emsp;[- Section 2:](#sec2) Explaining a prediction with exact cell-level Shapley values

&emsp;[- Section 3:](#sec3) Explaining performance with the neighborhood game

&emsp;[- Section 4:](#sec4) Selecting a coalition, pruning the backbone, and continuing training

In [1]:
from types import SimpleNamespace

import torch
from torch_geometric.nn.models import GCN

from topobench.nn.backbones.combinatorial.gccn import TopoTune
from topobench.explain import (
    CachedGame,
    CellMaskingGame,
    CellPlayer,
    CoalitionMaskedBackbone,
    anchored_top_k_mask,
    check_efficiency,
    explain_cells,
    greedy_mask,
    mask_to_coalition,
    prune_backbone_,
    sampled_shapley,
    shapley_values,
    top_k_mask,
)

## **Section 1:** A toy complex, a small TopoTune model, and a toy task <a class="anchor" id="sec1"></a>

Our complex is two triangles sharing an edge, lifted to a 2-complex: 4 nodes, 5 edges, 2 faces. It is small enough to enumerate games exactly, yet every neighborhood below is non-trivial on it.

Neighborhood matrices follow the TopoBench convention (sparse shape `[n_dst, n_src]`, indices `(dst_cell, src_cell)`), and neighborhood names follow the TopoTune grammar (see the TopoTune tutorial). We use the four neighborhoods of the default TopoTune config plus two more inter-rank routes, so both intra-rank and inter-rank message passing are in play. **The order of this list is the player (bit) order for every mask in this tutorial.**

In [2]:
# two triangles sharing edge (1, 2): faces A = {0, 1, 2}, B = {1, 2, 3}
EDGES = [(0, 1), (0, 2), (1, 2), (1, 3), (2, 3)]  # edge ids 0..4
FACES = [(0, 1, 2), (1, 2, 3)]                    # face ids 0..1
FACE_EDGES = {0: [0, 1, 2], 1: [2, 3, 4]}         # face -> its edges
N0, N1, N2 = len({n for e in EDGES for n in e}), len(EDGES), len(FACES)


def sparse(pairs, shape):
    idx = torch.tensor(sorted(set(pairs)), dtype=torch.long).T
    values = torch.ones(idx.shape[1])
    return torch.sparse_coo_tensor(idx, values, size=shape).coalesce()


edge_node = [(e, n) for e, ab in enumerate(EDGES) for n in ab]
edge_face = [(e, f) for f, es in FACE_EDGES.items() for e in es]
edge_coface = [
    (e, f) for es in FACE_EDGES.values() for e in es for f in es if e != f
]
node_via_face = [
    (a, b) for face in FACES for a in face for b in face if a != b
]

MATS = {
    # the default TopoTune config's four neighborhoods ...
    "up_adjacency-1": sparse(edge_coface, (N1, N1)),    # edge <-> edge (faces)
    "up_incidence-0": sparse(edge_node, (N1, N0)),      # nodes -> edges
    "down_incidence-2": sparse(edge_face, (N1, N2)),    # faces -> edges
    "2-up_adjacency-0": sparse(node_via_face, (N0, N0)),  # node <-> node (faces)
    # ... plus two more inter-rank routes
    "down_incidence-1": sparse([(n, e) for e, n in edge_node], (N0, N1)),  # edges -> nodes
    "up_incidence-1": sparse([(f, e) for e, f in edge_face], (N2, N1)),   # edges -> faces
}
NEIGHBORHOODS = list(MATS)  # player/bit order


def toy_batch(channels=8, seed=0):
    g = torch.Generator().manual_seed(seed)
    return SimpleNamespace(
        x_0=torch.randn(N0, channels, generator=g),
        x_1=torch.randn(N1, channels, generator=g),
        x_2=torch.randn(N2, channels, generator=g),
        cell_statistics=torch.tensor([[N0, N1, N2]], dtype=torch.long),
        **MATS,
    )

The model is a TopoTune backbone (one small GCN per neighborhood and layer) with a tiny graph-level readout on the node rank, producing one binary logit per complex.

In [3]:
CHANNELS = 8


class ToyModel(torch.nn.Module):
    def __init__(self, neighborhoods, channels=CHANNELS, layers=2):
        super().__init__()
        torch.manual_seed(1)
        gnn = GCN(
            in_channels=channels,
            hidden_channels=channels,
            out_channels=channels,
            num_layers=1,
        )
        self.backbone = TopoTune(
            GNN=gnn,
            neighborhoods=neighborhoods,
            layers=layers,
            use_edge_attr=False,
            activation="relu",
        )
        self.head = torch.nn.Linear(channels, 1)

    def forward(self, batch):
        x_per_rank = self.backbone(batch)  # {rank: features}
        return self.head(x_per_rank[0].mean(dim=0)).squeeze()  # one logit


model = ToyModel(NEIGHBORHOODS)

The toy task: each sample is the same complex with fresh random features, labeled by the sign of its mean node signal. Two things matter for what follows:

* **TopoTune's forward mutates its input batch** (it overwrites `batch.x_{rank}` with hidden states), so we always rebuild batches from their seed instead of reusing them. `CellMaskingGame` handles this automatically by snapshotting all ranks.
* Rebuilding batches deterministically from seeds makes every game below a deterministic function of the coalition mask, which is what memoization (`CachedGame`) assumes.

In [4]:
DATASET = [
    (seed, float(toy_batch(seed=seed).x_0.mean() > 0))
    for seed in range(100, 116)
]


def train(model, steps=60, lr=0.01):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = torch.nn.BCEWithLogitsLoss()
    model.train()
    for _ in range(steps):
        opt.zero_grad()
        loss = sum(
            loss_fn(model(toy_batch(seed=seed)), torch.tensor(label))
            for seed, label in DATASET
        ) / len(DATASET)
        loss.backward()
        opt.step()
    model.eval()
    return float(loss)


def accuracy(model):
    with torch.no_grad():
        correct = sum(
            float(torch.sigmoid(model(toy_batch(seed=seed))) > 0.5) == label
            for seed, label in DATASET
        )
    return correct / len(DATASET)


final_loss = train(model)
print(f"final training loss: {final_loss:.3f}, accuracy: {accuracy(model):.3f}")

final training loss: 0.008, accuracy: 1.000


## **Section 2:** Explaining a prediction with exact cell-level Shapley values <a class="anchor" id="sec2"></a>

We now explain one prediction of the trained model. Every cell of the complex is a player (11 in total), and the game value of a coalition is the model's logit with the features of all *absent* cells masked to zero. Only features are masked — the incidence structure stays intact — so the game asks "what if this cell's signal were absent?", not "what if the complex were rewired?".

With 11 players, `explain_cells` enumerates all $2^{11} = 2048$ coalitions and returns *exact* Shapley values (it switches to permutation sampling automatically beyond `EXACT_PLAYER_LIMIT` players). We also request the pairwise Shapley interaction indices of Grabisch & Roubens.

In [5]:
batch = toy_batch(seed=100)  # explain the model's output on this sample
players = (
    [CellPlayer(rank=0, index=i) for i in range(N0)]
    + [CellPlayer(rank=1, index=i) for i in range(N1)]
    + [CellPlayer(rank=2, index=i) for i in range(N2)]
)


def predicted_logit(b):
    with torch.no_grad():
        return model(b)


explanation = explain_cells(predicted_logit, batch, players, interactions=True)
print(f"exact: {explanation.exact}   model evaluations: {explanation.evaluations}\n")

rank_names = {0: "node", 1: "edge", 2: "face"}
for player, phi in zip(explanation.players, explanation.phi):
    print(f"{rank_names[player.rank]} {player.index}:  phi = {phi:+.4f}")

exact: True   model evaluations: 2048

node 0:  phi = -0.1404
node 1:  phi = -0.1113
node 2:  phi = -3.5534
node 3:  phi = +0.9140
edge 0:  phi = -0.7325
edge 1:  phi = -0.3131
edge 2:  phi = -0.9788
edge 3:  phi = +0.0321
edge 4:  phi = +0.4557
face 0:  phi = -0.8784
face 1:  phi = +0.0948


A Shapley explanation comes with axioms we can *check* instead of trusting: efficiency says the per-cell attributions must sum exactly to the gap between the full prediction and the fully-masked prediction.

In [6]:
game = CachedGame(
    n_players=len(players),
    evaluate=CellMaskingGame(predicted_logit, batch, players),
)
gap = check_efficiency(explanation.phi, game, len(players), atol=1e-4)
full_mask = (1 << len(players)) - 1
print(f"v(all cells) = {game(full_mask):+.4f}")
print(f"v(no cells)  = {game(0):+.4f}")
print(f"sum(phi)     = {explanation.phi.sum():+.4f}   (efficiency gap {gap:.1e})")

top_pair = max(explanation.interactions, key=lambda t: abs(explanation.interactions[t]))
i, j = sorted(top_pair)
pi, pj = players[i], players[j]
print(
    f"strongest pairwise interaction: {rank_names[pi.rank]} {pi.index} & "
    f"{rank_names[pj.rank]} {pj.index}: {explanation.interactions[top_pair]:+.4f}"
)

v(all cells) = -8.2026
v(no cells)  = -2.9914
sum(phi)     = -5.2113   (efficiency gap 8.9e-16)
strongest pairwise interaction: node 2 & node 3: -1.3336


## **Section 3:** Explaining performance with the neighborhood game <a class="anchor" id="sec3"></a>

The second question is architectural: how much does each *neighborhood* contribute to task performance? `CoalitionMaskedBackbone` wraps the trained backbone in place and drops, per layer, the contribution of every route whose bit is absent from the active coalition — exactly what "this player is absent" means for a message-passing route. The wrapper captures the backbone's neighborhood list at wrap time as `wrapped.players`; bit $i$ of every mask refers to `wrapped.players[i]`, whatever the length of the list.

The game value of a coalition is simply the model's accuracy on our toy dataset under that coalition. We wrap it in a `CachedGame` so repeated coalitions are free and the true cost stays visible.

In [7]:
wrapped = CoalitionMaskedBackbone(model.backbone)
n_players = len(wrapped.players)
print("players (bit order):", wrapped.players)


def coalition_accuracy(mask):
    with wrapped.coalition(mask):
        return accuracy(model)


game = CachedGame(n_players=n_players, evaluate=coalition_accuracy)
print(f"grand coalition accuracy: {game(wrapped.full_mask):.3f}")
print(f"empty coalition accuracy: {game(0):.3f}")

players (bit order): ['up_adjacency-1', 'up_incidence-0', 'down_incidence-2', '2-up_adjacency-0', 'down_incidence-1', 'up_incidence-1']
grand coalition accuracy: 1.000
empty coalition accuracy: 0.750


With 6 players the game has only $2^6 = 64$ coalitions, so we can afford the exact answer and use it to sanity-check the sampled estimator (which is the tool you would reach for with more neighborhoods, where $2^n$ model evaluations are out of reach). One sampling *pass* is one random permutation of the players; the estimator reports a per-player standard error and its true cost in distinct game evaluations.

In [8]:
att = sampled_shapley(game, n_players, passes=64, seed=0)
exact = shapley_values(game, n_players)

print(f"{'neighborhood':20s} {'sampled phi':>14s} {'exact phi':>10s}")
for name, phi_s, se, phi_e in zip(wrapped.players, att.phi, att.stderr, exact):
    print(f"{name:20s} {phi_s:+.4f} ± {se:.4f} {phi_e:+10.4f}")

print(f"\ndistinct game evaluations so far: {game.calls} (of 64 possible)")

neighborhood            sampled phi  exact phi
up_adjacency-1       +0.0391 ± 0.0108    +0.0365
up_incidence-0       +0.0723 ± 0.0151    +0.0573
down_incidence-2     +0.0332 ± 0.0085    +0.0312
2-up_adjacency-0     +0.0254 ± 0.0082    +0.0260
down_incidence-1     +0.0801 ± 0.0260    +0.0990
up_incidence-1       +0.0000 ± 0.0000    +0.0000

distinct game evaluations so far: 64 (of 64 possible)


One reading note: an attribution of *exactly* zero in an accuracy-scored game only means that no probed coalition's decisions flipped — the route may still move logits. To distinguish a coarse metric from genuinely dead wiring, run `find_null_players` on a logit-scored game: a route the architecture claims is live showing up as a null player there is a bug signal.

## **Section 4:** Selecting a coalition, pruning the backbone, and continuing training <a class="anchor" id="sec4"></a>

Attributions rank neighborhoods; a *selector* turns them into a coalition. Three selectors, in ascending order of robustness to redundancy:

* **top-k** by Shapley value — the obvious rule, but wrong under redundancy: perfect substitutes split their shared payoff and both get picked;
* **anchored top-k** — always keeps a designated anchor player (here the node-to-edge route, as an example of domain knowledge) and fills the rest by Shapley value;
* **greedy** on the game itself — after one of two substitutes is added, the other's marginal gain collapses, so redundancy is handled structurally.

In [9]:
k = 3
anchor_bit = wrapped.players.index("up_incidence-0")
selections = {
    "top-k": top_k_mask(exact, k),
    "anchored top-k": anchored_top_k_mask(exact, k, anchor_bit=anchor_bit),
    "greedy": greedy_mask(game, n_players, k),
}
for label, mask in selections.items():
    names = mask_to_coalition(mask, wrapped.players)
    print(f"{label:15s} mask={mask:2d}  accuracy={game(mask):.3f}  {names}")

top-k           mask=19  accuracy=1.000  ('up_adjacency-1', 'up_incidence-0', 'down_incidence-1')
anchored top-k  mask=19  accuracy=1.000  ('up_adjacency-1', 'up_incidence-0', 'down_incidence-1')
greedy          mask=19  accuracy=1.000  ('up_adjacency-1', 'up_incidence-0', 'down_incidence-1')


Finally we commit to the greedy coalition: `prune_backbone_` removes the dropped routes and their per-layer GNN copies *in place*, keeping the trained weights of the kept routes. The pruned model therefore warm-starts: we continue training for a few steps and it pays compute only for the neighborhoods it kept.

In [10]:
chosen = selections["greedy"]
kept = prune_backbone_(model.backbone, chosen)
n_params = sum(p.numel() for p in model.parameters())
print(f"kept neighborhoods: {kept}")
print(f"routes per layer: {len(model.backbone.graph_routes[0])}, parameters: {n_params}")

acc_after_prune = accuracy(model)
train(model, steps=20)
print(f"accuracy after pruning: {acc_after_prune:.3f}")
print(f"accuracy after 20 more training steps: {accuracy(model):.3f}")

kept neighborhoods: ['up_adjacency-1', 'up_incidence-0', 'down_incidence-1']
routes per layer: 3, parameters: 441


accuracy after pruning: 1.000
accuracy after 20 more training steps: 1.000


## Where to go from here

* One warning before you wrap a real model: `CellMaskingGame` masks the per-rank feature matrices `x_{rank}`, so it only explains models that *read* them. Models that consume per-hop encodings — HOPSE assembles its backbone input from the `x{rank}_{hop}` tensors written by its feature encoder and never reads `x_{rank}` — make the game a structural no-op: `v(full) == v(empty)` exactly and every Shapley value is 0. Use `HopseCellMaskingGame` for them, which encodes once and masks the encoded per-hop rows the model actually consumes.
* `topobench.explain.axioms` has more executable checks: `find_null_players` flags players whose marginal contribution is zero on every probed coalition (a live route showing up there signals dead wiring), `check_symmetry_pair` quantifies substitutes, and `check_batch_invariance` guards against models whose per-sample outputs depend on batch composition.
* `topobench.explain.selection` also has automatic-size rules (`threshold_at_zero`, `cost_regularized`, `greedy_with_noise_stop`, `smallest_sufficient_coalition`) and `submodularity_census`, which measures whether the greedy selector's $(1 - 1/e)$ guarantee applies to your game instead of assuming it.
* `resample_pick_stability` re-runs the sampled estimator many times and reports how often the selection agrees with itself — a persistent split is the signature of near-substitute neighborhoods, not sampling noise.
* Everything here applies to real TopoBench models: wrap the backbone of a trained `TBModel`, score coalitions with your validation metric, and prune.